In [2]:
import os
import numpy as np
import pandas as pd

Now im going to run ADF test on these stocks to see if they are stationary or not

In [ ]:
selected_tickers = [ "GARAN.IS", "AKBNK.IS", "ISCTR.IS", "YKBNK.IS", "TUPRS.IS", 
    "EREGL.IS", "KCHOL.IS", "SAHOL.IS", "SISE.IS", "THYAO.IS", "BIMAS.IS" ]

print("--- Results ---")

for ticker_name in selected_tickers:
    ticker_name = ticker_name.replace(".IS", "")
    df = pd.read_csv(f"../data/{ticker_name}.csv")

    df['daily_change'] = df['Close'].diff()
    df['lag'] = df['Close'].shift(1)

    df_clean = df.dropna(subset=['daily_change', 'lag'])

    ones_col = np.ones(len(df_clean))
    matrix = np.column_stack((ones_col, df_clean['lag'].values))
    matrix_t = matrix.T

    matrix_multiplication = matrix_t @ matrix
    inv_matrix_multiplication = np.linalg.inv(matrix_multiplication)

    y = df_clean['daily_change'].values
    coefficients = inv_matrix_multiplication @ (matrix_t  @ y)

    intercept = coefficients[0] #Drift
    gamma = coefficients[1] #Gamma in ADF test

    y_pred = matrix @ coefficients #Predicting data according to our model
    residuals = y - y_pred  #Finding the error between the predicted data nad our real data

    n = len(y)
    degrees_of_freedom = n - 2
    residual_variance = np.sum(residuals ** 2) / degrees_of_freedom

    vcov = residual_variance * inv_matrix_multiplication

    se_gamma = np.sqrt(vcov[1,1])

    adf_stat = coefficients[1] / se_gamma

    print(f"Results for {ticker_name}: ")
    print(f"Gamma (γ): {coefficients[1]:.6f}")  
    print(f"Gamma Standard: {se_gamma:.6f}")
    print(f"ADF Test Statistic : {adf_stat:.6f}") #Lower than -2.86 is stationary, greater than -2.86 non-stationary
    print("\n")

--- SONUÇLAR ---
Gamma (γ) Katsayısı : -0.001378
Gamma Standart Hatası : 0.001403
ADF Test İstatistiği  : -0.982356
